# American Dialect Classification with Wav2Vec and Convolutional Neural Networks

Previously, I used wav2vec embeddings with a logistic regression classifier and hit a performance ceiling of around 40% accuracy. That approach relied on the assumption that temporal structure is not important, and that all relevant information can be captured through global statistics (i.e, statistics pooling). Before moving on from this setup, I wanted to test that assumption by replacing the logistic regression classifier with a model capable of learning local temporal patterns, such as a CNN.

I used the TIMIT dataset, summarized below:
| Region # | Region name              | # Speakers |
|----------|--------------------------|------------|
| DR1      | New England              | 49         |
| DR2      | Northern                 | 102        |
| DR3      | North Midland            | 102        |
| DR4      | South Midland            | 100        |
| DR5      | Southern                 | 98         |
| DR6      | New York City            | 46         |
| DR7      | Western                  | 100        |

In [1]:
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2Model, logging
import torch
import torchaudio
import glob
import numpy as np

DATA_PATH = "../data/raw/TIMIT/data"
N_REGIONS = 7
utterances = ["SA1", "SA2"]

logging.set_verbosity_error()

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("facebook/wav2vec2-base")
model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base", output_hidden_states=True)
model.eval()  # inference mode

None

/home/rose/anaconda3/envs/dialect/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 211/211 [00:00<00:00, 15394.75it/s]


In [2]:
import pandas as pd

paths = glob.glob(f'{DATA_PATH}/*/*/*/*.WAV')

def parse_path(path):

    path_split = path.split("/")
    data_split = path_split[5]
    dialect = path_split[6]
    speaker_id = path_split[7]
    utt = path_split[8].replace(".WAV", "")

    return {"split": data_split,
            "dialect": dialect,
            "speaker_id": speaker_id,
            "utterance": utt,
            "filepath": path}

rows = [parse_path(path) for path in paths]

df = pd.DataFrame(rows)
df = df[(df["utterance"].isin(utterances)) & (df["dialect"] != "DR8")]

df

,split,dialect,speaker_id,utterance,filepath
3,TRAIN,DR3,MRBC0,SA1,../data/raw/TIMIT/data/TRAIN/DR3/MRBC0/SA1.WAV
6,TRAIN,DR3,MRBC0,SA2,../data/raw/TIMIT/data/TRAIN/DR3/MRBC0/SA2.WAV
15,TRAIN,DR3,FCKE0,SA1,../data/raw/TIMIT/data/TRAIN/DR3/FCKE0/SA1.WAV
17,TRAIN,DR3,FCKE0,SA2,../data/raw/TIMIT/data/TRAIN/DR3/FCKE0/SA2.WAV
25,TRAIN,DR3,MCEF0,SA1,../data/raw/TIMIT/data/TRAIN/DR3/MCEF0/SA1.WAV
...,...,...,...,...,...
6166,TEST,DR2,FCMR0,SA2,../data/raw/TIMIT/data/TEST/DR2/FCMR0/SA2.WAV
6174,TEST,DR2,MTAS1,SA1,../data/raw/TIMIT/data/TEST/DR2/MTAS1/SA1.WAV
6177,TEST,DR2,MTAS1,SA2,../data/raw/TIMIT/data/TEST/DR2/MTAS1/SA2.WAV
6184,TEST,DR2,MMDB1,SA1,../data/raw/TIMIT/data/TEST/DR2/MMDB1/SA1.WAV


In [31]:
def extract_embedding(path):

    # load wav
    y, sr = torchaudio.load(path)

    # TIMIT sr should be 16kHz, but resample if not
    if sr != 16000: 
        y = torchaudio.transforms.Resample(sr, 16000)(y)
    y = y.squeeze(0)  # remove channel dim if mono, model expects 1D waveform

    # get wav2vec embedding
    inputs = feature_extractor(y, sampling_rate=16000, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    hidden_states = outputs.hidden_states
    
    # get middle 4 layers
    layers = hidden_states[6:10]
    layer_embeddings = []

    # average layers
    embedding = torch.stack(layers, dim=0) # (4, B, T, 768)
    embedding = embedding.mean(dim=0) # (B, T, 768)

    embedding = embedding.squeeze(0) # (T, 768)
    embedding = embedding.T # (768, T)
    
    return embedding

def build_embeddings(df):
    embeddings = []

    for path in df["filepath"]:
        emb = extract_embedding(path)
        embeddings.append(emb)
    
    return embeddings

def get_labels(df):
    y = df["dialect"].str[2].astype(int) - 1

    return y

X = build_embeddings(df)
y = get_labels(df)

In [45]:
def get_split(X, y, df, split):
    mask = df["split"] == split.upper()

    X_split = [x for x, m in zip(X, mask) if m]
    y_split = y[mask].tolist() # series -> list

    return X_split, y_split

X_train, y_train = get_split(X, y, df, "train")
X_test, y_test   = get_split(X, y, df, "test")

# len should be (n_speakers * 2) (n_speakers * 2)
print(f'X_train length: {len(X_train)} | y_train length: {len(y_train)}')

X_train length: 880 | y_train length: 880


###  CNN

In [98]:
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader

class DialectDataset(Dataset):
    def __init__(self, X: list, y: int):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.y[i]


def collate_fn(batch):
    xs, ys = zip(*batch)

    # pad frames
    xs = [x.T for x in xs]
    xs = pad_sequence(xs, batch_first=True)

    xs = xs.transpose(1, 2) # (B, 786, T_max)
    ys = torch.tensor(ys)

    return xs, ys

In [99]:

train_dataset = DialectDataset(X_train, y_train)
test_dataset  = DialectDataset(X_test, y_test)

train_loader = DataLoader(train_dataset,
                          batch_size=16,
                          shuffle=True,
                          collate_fn=collate_fn)

test_loader = DataLoader(test_dataset,
                         batch_size=16,
                         shuffle=False,
                         collate_fn=collate_fn)

In [119]:
import torch.nn as nn

class CNN(nn.Module):
    def __init__(self, n_classes=7):
        super().__init__()
        
        self.conv1 = nn.Conv1d(in_channels=768,
                               out_channels=256,
                               kernel_size=5,
                               padding=2)

        self.conv2 = nn.Conv1d(in_channels=256,
                               out_channels=256,
                               kernel_size=5,
                               padding=2)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(256 * 2, n_classes) # * 2 for stats pooling

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.dropout(x)
        x = self.relu(self.conv2(x)) # (B, 256, T)
        x = self.dropout(x)

        # stats pooling
        mean = x.mean(dim=2)
        std  = x.std(dim=2)
        x = torch.cat([mean, std], dim=1) # (B, 512)

        return self.fc(x)

In [120]:

def train_model(model, train_loader, test_loader, epochs=20, lr=1e-3, device="cpu"):
    
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):

        # Train
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0

        for X, y in train_loader:
            X = X.to(device) # (B, 768, T)
            y = y.to(device) # (B,)

            optimizer.zero_grad()
            logits = model(X) # (B, num_classes)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            train_correct += (preds == y).sum().item()
            train_total += y.size(0)

        train_acc = train_correct / train_total

        # Eval
        model.eval()
        test_correct, test_total, test_loss = 0, 0, 0

        with torch.no_grad():
            for X, y in test_loader:
                X = X.to(device)
                y = y.to(device)

                logits = model(X)
                loss = criterion(logits, y)

                test_loss += loss.item()
                
                preds = torch.argmax(logits, dim=1)
                test_correct += (preds == y).sum().item()
                test_total += y.size(0)

        test_acc = test_correct / test_total

        # Logging
        print(f'Epoch {epoch + 1}/{epochs} | '
              f'Train Acc: {train_acc:.3f} | '
              f'Test Acc: {test_acc:.3f} | '
              f'Train Loss: {train_loss:.3f}')

In [122]:
cnn = CNN(N_REGIONS)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(cnn.parameters(), lr=1e-4)

train_model(cnn, train_loader, test_loader, 15)

Epoch 1/15 | Train Acc: 0.182 | Test Acc: 0.188 | Train Loss: 104.633
Epoch 2/15 | Train Acc: 0.244 | Test Acc: 0.354 | Train Loss: 93.901
Epoch 3/15 | Train Acc: 0.388 | Test Acc: 0.315 | Train Loss: 81.388
Epoch 4/15 | Train Acc: 0.392 | Test Acc: 0.344 | Train Loss: 79.279
Epoch 5/15 | Train Acc: 0.483 | Test Acc: 0.360 | Train Loss: 70.711
Epoch 6/15 | Train Acc: 0.545 | Test Acc: 0.350 | Train Loss: 64.631
Epoch 7/15 | Train Acc: 0.570 | Test Acc: 0.392 | Train Loss: 59.416
Epoch 8/15 | Train Acc: 0.601 | Test Acc: 0.385 | Train Loss: 55.343
Epoch 9/15 | Train Acc: 0.631 | Test Acc: 0.401 | Train Loss: 49.520
Epoch 10/15 | Train Acc: 0.667 | Test Acc: 0.395 | Train Loss: 45.265
Epoch 11/15 | Train Acc: 0.719 | Test Acc: 0.376 | Train Loss: 40.068
Epoch 12/15 | Train Acc: 0.717 | Test Acc: 0.404 | Train Loss: 38.502
Epoch 13/15 | Train Acc: 0.791 | Test Acc: 0.379 | Train Loss: 31.041
Epoch 14/15 | Train Acc: 0.802 | Test Acc: 0.363 | Train Loss: 30.415
Epoch 15/15 | Train Acc: 0.8

### References

1. Garofolo, John S., et al. TIMIT Acoustic-Phonetic Continuous Speech Corpus LDC93S1. Web 
Download. Philadelphia: Linguistic Data Consortium, 1993.